# 串行中断
## 跨节点串行中断

In [10]:
from typing import TypedDict
from rich import print
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt


# 声明状态
class State(TypedDict):
    username: str
    age: str
    profile: str


# 声明节点：串行路径上每个节点各带一个 interrupt
def ask_name(state: State) -> dict:
    print("========= 执行 ask_name 节点 =========")
    username = interrupt("请输入你的名字")  # 副作用必须放在 interrupt() 之后：恢复时节点从头重跑
    return {"username": username}


def ask_age(state: State) -> dict:
    print("========= 执行 ask_age 节点 =========")
    age = interrupt("请输入你的年龄")
    return {"age": age}


def summarize(state: State) -> dict:
    return {"profile": f"{state['username']}，{state['age']}岁"}


# 构建图结构：ask_name -> ask_age -> summarize 串行链
builder = StateGraph(state_schema=State)
builder.add_node("ask_name", ask_name)
builder.add_node("ask_age", ask_age)
builder.add_node("summarize", summarize)
builder.add_edge(START, "ask_name")
builder.add_edge("ask_name", "ask_age")
builder.add_edge("ask_age", "summarize")
builder.add_edge("summarize", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行 -> 触发第 1 次中断：ask_name 挂起，__interrupt__ 长度为 1
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)

# 注意：同一节点内写多个 interrupt() 也构成串行中断，但恢复值按出现顺序「索引匹配」，
# 跨节点写法（本案例）更直观也更安全


========= 执行 ask_name 节点 =========

{'__interrupt__': [Interrupt(value='请输入你的名字', id='f35a522d2feef8237b3d1bbda0feb6e2')]}

In [11]:
# 串行恢复：一次只应答当前挂起的那一个断点；恢复后图继续执行，
# 走到 ask_age 再次中断 —— while 循环逐个应答，直到图走完
# （ask_name 恢复时从头重跑，interrupt() 直接返回恢复值，不会再次挂起）
while res.get("__interrupt__"):
    ask_msg = res["__interrupt__"][0].value
    answer = input(ask_msg)
    res = graph.invoke(Command(resume=answer), config=config)
    print(res)

========= 执行 ask_name 节点 =========

========= 执行 ask_age 节点 =========

{
    'username': 'aihaipeng',
    '__interrupt__': [Interrupt(value='请输入你的年龄', id='4da2787284232ee678cbc05b9486a606')]
}

========= 执行 ask_age 节点 =========

{'username': 'aihaipeng', 'age': '26', 'profile': 'aihaipeng，26岁'}

## 单节点串行中断

In [8]:
from typing import TypedDict
from rich import print
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt


# 声明状态
class State(TypedDict):
    username: str
    age: str


# 声明节点
def ask_user_info(state: State) -> dict:
    print("========= 执行 ask_user_info 节点 =========")
    username = interrupt("请输入你的名字")
    age = interrupt("请输入你的年龄")
    return {
        "username": username,
        "age": age
    }


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("ask_user_info", ask_user_info)
builder.add_edge(START, "ask_user_info")
builder.add_edge("ask_user_info", END)

# 设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)

========= 执行 ask_user_info 节点 =========

{'__interrupt__': [Interrupt(value='请输入你的名字', id='70c715c3e56fac2f6c0580a1055604f8')]}

In [9]:
while res.get("__interrupt__"):
    ask_msg = res["__interrupt__"][0].value
    answer = input(ask_msg)
    res = graph.invoke(Command(resume=answer), config=config)
    print(res)

========= 执行 ask_user_info 节点 =========

{'__interrupt__': [Interrupt(value='请输入你的年龄', id='70c715c3e56fac2f6c0580a1055604f8')]}

========= 执行 ask_user_info 节点 =========

{'username': 'aihaipeng', 'age': '26'}